# 第23章 无监督学习与降维

这个 notebook 用一个教学版 HR 图数据集演示：PCA 如何概括主要变化方向，$k$-means 如何在没有标签时做强制分群，DBSCAN 如何把稀疏离群点标成噪声候选。

## 学习目标

- 理解无监督学习不依赖显式标签
- 看到标准化如何改变 PCA 和聚类结果
- 理解 PCA 在做什么，而不只是会调用函数
- 比较 $k$-means 和 DBSCAN 的结构假设
- 把离群候选重新落回样本解释

In [ ]:
from __future__ import annotations

import csv
import math
import platform
from collections import Counter
from pathlib import Path

DATA_PATH = Path('../../data/small/stellar_hr_unsupervised_demo.csv').resolve()

with DATA_PATH.open(newline='', encoding='utf-8') as handle:
    rows = []
    for raw in csv.DictReader(handle):
        rows.append({
            'source_id': raw['source_id'],
            'label': raw['label'],
            'bp_rp': float(raw['bp_rp']),
            'absolute_g_mag': float(raw['absolute_g_mag']),
            'reference_group': raw['reference_group'],
        })

print(f'Loaded {len(rows)} HR-diagram samples from {DATA_PATH.name}')
print('reference groups:', dict(Counter(row['reference_group'] for row in rows)))
print('first sample:', rows[0])
print('Python version:', platform.python_version())


## 先标准化特征

无监督学习对尺度很敏感，所以我们先把颜色和绝对星等都转到可比较的尺度。

In [ ]:
feature_names = ['bp_rp', 'absolute_g_mag']
means = {}
stds = {}
for name in feature_names:
    values = [row[name] for row in rows]
    mean_value = sum(values) / len(values)
    variance = sum((value - mean_value) ** 2 for value in values) / len(values)
    means[name] = mean_value
    stds[name] = math.sqrt(variance) or 1.0

standardized_rows = []
for row in rows:
    standardized = {
        name: (row[name] - means[name]) / stds[name]
        for name in feature_names
    }
    standardized_rows.append({**row, **standardized})

print('means:', {name: round(value, 3) for name, value in means.items()})
print('stds:', {name: round(value, 3) for name, value in stds.items()})
print('peculiar source standardized coordinates:', {
    name: round(next(row[name] for row in standardized_rows if row['source_id'] == 'PX01'), 3)
    for name in feature_names
})


## 手写 PCA：看主要变化方向

这里我们用二维协方差矩阵的解析形式做一个最小 PCA，不依赖外部科学库。

In [ ]:
def principal_components_2d(data_rows):
    xs = [row['bp_rp'] for row in data_rows]
    ys = [row['absolute_g_mag'] for row in data_rows]
    cov_xx = sum(x * x for x in xs) / len(xs)
    cov_xy = sum(x * y for x, y in zip(xs, ys)) / len(xs)
    cov_yy = sum(y * y for y in ys) / len(xs)
    trace = cov_xx + cov_yy
    root = math.sqrt((cov_xx - cov_yy) ** 2 + 4.0 * cov_xy * cov_xy)
    lambda1 = 0.5 * (trace + root)
    lambda2 = 0.5 * (trace - root)

    if abs(cov_xy) > 1e-12:
        vector1 = [cov_xy, lambda1 - cov_xx]
    else:
        vector1 = [1.0, 0.0]
    norm = math.sqrt(vector1[0] ** 2 + vector1[1] ** 2)
    vector1 = [component / norm for component in vector1]
    vector2 = [-vector1[1], vector1[0]]
    return (lambda1, lambda2), vector1, vector2

eigenvalues, pc1_vector, pc2_vector = principal_components_2d(standardized_rows)
explained_ratio = eigenvalues[0] / sum(eigenvalues)

projected_rows = []
for row in standardized_rows:
    pc1 = row['bp_rp'] * pc1_vector[0] + row['absolute_g_mag'] * pc1_vector[1]
    pc2 = row['bp_rp'] * pc2_vector[0] + row['absolute_g_mag'] * pc2_vector[1]
    projected_rows.append({**row, 'pc1': pc1, 'pc2': pc2})

print('PCA eigenvalues:', [round(value, 3) for value in eigenvalues])
print('PC1 explained variance ratio:', round(explained_ratio, 3))
print('PC1 direction:', [round(value, 3) for value in pc1_vector])
for row in projected_rows[:5]:
    print(row['source_id'], row['reference_group'], round(row['pc1'], 3), round(row['pc2'], 3))


## $k$-means：强制分群

这里我们用固定种子做一个最小 $k$-means，观察算法如何把每个样本都塞进某个簇。

In [ ]:
def squared_distance(point, center):
    return sum((point[index] - center[index]) ** 2 for index in range(len(point)))

def run_kmeans(data_rows, seed_ids, iterations=20):
    centers = [
        [
            next(row['bp_rp'] for row in data_rows if row['source_id'] == source_id),
            next(row['absolute_g_mag'] for row in data_rows if row['source_id'] == source_id),
        ]
        for source_id in seed_ids
    ]

    for _ in range(iterations):
        groups = [[] for _ in centers]
        for row in data_rows:
            point = [row['bp_rp'], row['absolute_g_mag']]
            distances = [squared_distance(point, center) for center in centers]
            cluster_index = min(range(len(distances)), key=distances.__getitem__)
            groups[cluster_index].append(row)

        new_centers = []
        for center, group in zip(centers, groups):
            if not group:
                new_centers.append(center)
                continue
            new_centers.append([
                sum(item['bp_rp'] for item in group) / len(group),
                sum(item['absolute_g_mag'] for item in group) / len(group),
            ])

        movement = sum(squared_distance(old, new) for old, new in zip(centers, new_centers))
        centers = new_centers
        if movement < 1e-10:
            break

    assignments = {}
    inertia = 0.0
    for row in data_rows:
        point = [row['bp_rp'], row['absolute_g_mag']]
        distances = [squared_distance(point, center) for center in centers]
        cluster_index = min(range(len(distances)), key=distances.__getitem__)
        assignments[row['source_id']] = cluster_index
        inertia += distances[cluster_index]

    return centers, assignments, inertia

seed_ids = ['WD01', 'RG05', 'MS04']
centers, assignments, inertia = run_kmeans(projected_rows, seed_ids=seed_ids)
print('k-means seed ids:', seed_ids)
print('k-means inertia:', round(inertia, 3))
print('cluster centers:', [[round(value, 3) for value in center] for center in centers])

cluster_summary = {}
for row in projected_rows:
    cluster_index = assignments[row['source_id']]
    cluster_summary.setdefault(cluster_index, {})
    group = row['reference_group']
    cluster_summary[cluster_index][group] = cluster_summary[cluster_index].get(group, 0) + 1
print('cluster composition:', cluster_summary)
print('peculiar source assigned cluster:', assignments['PX01'])


## DBSCAN：允许噪声候选

现在换成密度视角，看看异常目标是否会更自然地被分离出来。

In [ ]:
UNVISITED = 0
NOISE = -1

def euclidean_distance_rows(left_row, right_row):
    return math.sqrt(
        (left_row['bp_rp'] - right_row['bp_rp']) ** 2
        + (left_row['absolute_g_mag'] - right_row['absolute_g_mag']) ** 2
    )

def dbscan_neighbors(data_rows, row, eps):
    return [other for other in data_rows if euclidean_distance_rows(row, other) <= eps]

def run_dbscan(data_rows, eps, min_samples):
    labels = {row['source_id']: UNVISITED for row in data_rows}
    cluster_id = 0
    for row in data_rows:
        source_id = row['source_id']
        if labels[source_id] != UNVISITED:
            continue
        neighbor_rows = dbscan_neighbors(data_rows, row, eps)
        if len(neighbor_rows) < min_samples:
            labels[source_id] = NOISE
            continue
        cluster_id += 1
        labels[source_id] = cluster_id
        seed_rows = [neighbor for neighbor in neighbor_rows if neighbor['source_id'] != source_id]
        index = 0
        while index < len(seed_rows):
            candidate = seed_rows[index]
            candidate_id = candidate['source_id']
            if labels[candidate_id] == NOISE:
                labels[candidate_id] = cluster_id
            if labels[candidate_id] != UNVISITED:
                index += 1
                continue
            labels[candidate_id] = cluster_id
            candidate_neighbors = dbscan_neighbors(data_rows, candidate, eps)
            if len(candidate_neighbors) >= min_samples:
                known_ids = {item['source_id'] for item in seed_rows}
                for neighbor in candidate_neighbors:
                    if neighbor['source_id'] not in known_ids:
                        seed_rows.append(neighbor)
                        known_ids.add(neighbor['source_id'])
            index += 1
    return labels

dbscan_labels = run_dbscan(projected_rows, eps=0.7, min_samples=2)
dbscan_summary = {}
for row in projected_rows:
    label = dbscan_labels[row['source_id']]
    dbscan_summary.setdefault(label, {})
    group = row['reference_group']
    dbscan_summary[label][group] = dbscan_summary[label].get(group, 0) + 1

noise_candidates = [row['source_id'] for row in projected_rows if dbscan_labels[row['source_id']] == NOISE]
print('DBSCAN cluster composition:', dbscan_summary)
print('noise candidates:', noise_candidates)


## Algorithm Understanding Bridge

无监督学习最容易被误读成“算法发现了物理类别”。这一段把 PCA、聚类和可视化降维分别写成结构假设，作为后续命名 cluster 前的检查材料。

In [ ]:
RESULTS_DIR = Path('results/part3_algorithm_understanding')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

cluster_lines = []
for cluster_id in sorted(cluster_summary):
    cluster_lines.append(f'- cluster {cluster_id}: {cluster_summary[cluster_id]}')

noise_line = ', '.join(noise_candidates) if noise_candidates else 'none in this setting'

unsupervised_bridge = f'''# Ch23 Algorithm Understanding Bridge

## PCA

- Structural hypothesis: the largest-variance directions are useful summaries of the data.
- What is optimized: orthogonal directions that maximize projected variance.
- What is learned: principal component directions and explained variance.
- Current PC1 explained variance ratio: {explained_ratio:.3f}.
- Caution: high explained variance is not the same as physical importance.

## k-means

- Structural hypothesis: samples are well represented by nearby cluster centers.
- Objective: minimize within-cluster squared distance to centers.
- What is fixed by the user: k, initialization, distance scale, stopping rule.
- Failure pattern: elongated structures or unequal-density groups may be forced into spherical-ish clusters.
- Current cluster composition by reference group:
{chr(10).join(cluster_lines)}

## DBSCAN

- Structural hypothesis: meaningful groups are density-connected regions.
- What is fixed by the user: eps and min_samples.
- Output roles: clustered points and noise candidates.
- Current noise candidates: {noise_line}.
- Failure pattern: one eps may not work when densities differ across the diagram.

## Gaussian Mixture Model

- Structural hypothesis: data can be approximated by overlapping probability components.
- What is learned: component means, covariances, and mixture weights.
- Useful idea: soft assignment can express ambiguous membership better than hard cluster labels.

## t-SNE / UMAP

- Role: visualization and neighborhood exploration.
- Boundary: a visually separated island is not automatically a physical population.

## Required Naming Statement

Any cluster name should start as: algorithmic group, not physical class unless externally validated.
'''

output_path = RESULTS_DIR / 'ch23_algorithm_understanding.md'
output_path.write_text(unsupervised_bridge, encoding='utf-8')
print('wrote', output_path)
print('DBSCAN noise candidates:', noise_candidates)


## 小结

这个例子说明：无监督学习最适合做结构探索和候选整理。$k$-means 会强制每个样本归入某簇，而 DBSCAN 更愿意承认“这点不属于主体结构”。

In [ ]:
snapshot = {
    'dataset': DATA_PATH.name,
    'n_rows': len(rows),
    'pc1_explained_ratio': round(explained_ratio, 3),
    'kmeans_inertia': round(inertia, 3),
    'dbscan_noise_candidates': noise_candidates,
    'python_version': platform.python_version(),
}

print('Unsupervised snapshot:')
for key, value in snapshot.items():
    print(f'  {key}: {value}')
